In [ ]:
import subprocess,sys
subprocess.check_call([sys.executable,'-m','pip','install','-q','pandas==2.2.3','numpy==2.3.5','pyarrow==26.0.0','holidays==0.106','catboost==1.2.10','lightgbm==4.7.0','xgboost==3.1.3','scipy==1.16.3','scikit-learn==1.6.1'])


In [ ]:
from pathlib import Path
p=Path('/kaggle/working/reserved_launcher.py')
p.write_text('"""Evaluate a frozen choice, then train March; never select using reserved errors."""\nimport base64,gc,hashlib,io,json,os,sys,zipfile,warnings,importlib.metadata\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nPAYLOAD=\'...\'\nBACKEND=\'cuda\'\nCPU_GROUP=\'main\'\n\ndef main():\n    root=Path(\'/tmp/x5_reserved_\'+BACKEND);parent=root/\'Forecasting\';here=parent/\'research_stage2\';here.mkdir(parents=True,exist_ok=True)\n    with zipfile.ZipFile(io.BytesIO(base64.b64decode(PAYLOAD))) as z:\n        for name in z.namelist():\n            if name.endswith(\'.py\') or name==\'frozen_final_selection.json\':\n                target=(parent if name in [\'notebook_api.py\',\'run_validation.py\'] else here)/name;target.write_bytes(z.read(name))\n        stores=pd.read_parquet(io.BytesIO(z.read(\'stores.parquet\')));locations=pd.read_parquet(io.BytesIO(z.read(\'locations.parquet\')))\n        old_manifest=json.loads(z.read(\'old_manifest.json\'));expected=json.loads(z.read(\'expected_legacy.json\'))\n        spatial_source=z.read(\'spatial_source.txt\').decode();research_source=z.read(\'research_source.txt\').decode()\n        from scipy import sparse\n        old_graphs={kind:sparse.load_npz(io.BytesIO(z.read(\'preserved_\'+kind+\'_weights.npz\'))) for kind in [\'knn8\',\'hybrid8\']}\n    source=list(Path(\'/kaggle/input\').rglob(\'input_manifest.json\'))[0].parent\n    derived=list(Path(\'/kaggle/input\').rglob(\'research_manifest.json\'))[0].parent\n    prepared=list(Path(\'/kaggle/input\').rglob(\'prepared_stage2.zip\'));assert len(prepared)==1\n    raw_dir=root/\'Data/Shops\';raw_dir.mkdir(parents=True,exist_ok=True);baseline=parent/\'baseline\';baseline.mkdir(exist_ok=True)\n    (raw_dir/\'train_2.csv\').symlink_to(source/\'train_2.csv\');(baseline/\'original_solution.ipynb\').symlink_to(source/\'original_solution.ipynb\')\n    sys.path.insert(0,str(parent));sys.path.insert(0,str(here))\n    cache=parent/\'cache\';cache.mkdir(exist_ok=True);new_cache=here/\'cache\';new_cache.mkdir(exist_ok=True)\n    with zipfile.ZipFile(prepared[0]) as z:\n        assert z.testzip() is None\n        for name in z.namelist():\n            target=new_cache/\'features.parquet\' if name==\'stage2_features.parquet\' else cache/name\n            assert target.resolve().is_relative_to(root);target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(z.read(name))\n    from notebook_api import read_raw,load_recipe,compact_fold,NOTEBOOK_SHA256,MODELS\n    from features import compute\n    from experiment import execute\n    recipe=load_recipe();raw=read_raw(recipe);new,manifest=compute(raw,stores,locations)\n    pd.testing.assert_frame_equal(new,pd.read_parquet(new_cache/\'features.parquet\'),check_exact=True)\n    digest=hashlib.sha256((here/\'features.py\').read_bytes()).hexdigest();manifest[\'implementation_sha256\']=digest\n    (new_cache/\'feature_manifest.json\').write_text(json.dumps(manifest,indent=2))\n    (here/\'feature_checks.json\').write_text(json.dumps(dict(status=\'passed\',implementation_sha256=digest,canonical_feature_exact=True),indent=2))\n    import geo_control\n    geo_control.compute(raw,stores,locations).to_parquet(new_cache/\'geo_control.parquet\',index=False)\n    from sklearn.neighbors import BallTree\n    radians=np.radians(locations.sort_values(\'node\')[[\'settlement_lat\',\'settlement_lon\']].to_numpy())\n    distances,neighbours=BallTree(radians,metric=\'haversine\').query(radians,k=33)\n    np.savez_compressed(Path(\'/kaggle/working\')/(\'canonical_graph_\'+BACKEND+\'.npz\'),radians=radians,distances_rad=distances,neighbours=neighbours)\n    (here/\'geo_control_checks.json\').write_text(json.dumps(dict(status=\'passed\',implementation_sha256=hashlib.sha256((here/\'geo_control.py\').read_bytes()).hexdigest()),indent=2))\n    old_dir=parent/\'research/cache\';old_dir.mkdir(parents=True,exist_ok=True)\n    columns=list(dict.fromkeys(c for group in old_manifest[\'groups\'].values() for c in group))\n    old=pd.concat([pd.read_parquet(derived/name,columns=[\'new_id\',\'year_month_index\']+columns)\n        for name in [\'2025-02_train.parquet\',\'2025-02_holdout.parquet\']],ignore_index=True)\n    old.to_parquet(old_dir/\'features.parquet\',index=False);stores.to_parquet(old_dir/\'stores.parquet\',index=False)\n    (old_dir/\'feature_manifest.json\').write_text(json.dumps(old_manifest,indent=2))\n    from extend_old import preserved_functions,extend\n    namespace=preserved_functions(spatial_source,research_source,stores,locations)\n    original_family=namespace[\'weight_family\']\n    namespace[\'weight_family\']=lambda locations,kind,profile:old_graphs[kind] if kind in old_graphs else original_family(locations,kind,profile)\n    legacy,_=namespace[\'spatial_table\'](raw,kind=\'exp50\')\n    fingerprint=hashlib.sha256(pd.util.hash_pandas_object(legacy[expected[\'columns\']],index=False).values.tobytes()).hexdigest()\n    assert fingerprint==expected[\'sha256\'];legacy.to_parquet(cache/\'spatial_exp50.parquet\',index=False)\n    future,checks=extend(raw,stores,locations,old,old_manifest[\'groups\'],namespace)\n    future.to_parquet(new_cache/\'old_future.parquet\',index=False)\n    frozen=json.loads((here/\'frozen_final_selection.json\').read_text())\n    models=[MODELS[-1]] if BACKEND==\'cuda\' else (MODELS[:2] if CPU_GROUP==\'main\' else [MODELS[2]])\n    del raw,new,legacy,old,future;gc.collect()\n    out=Path(\'/kaggle/working\');tag=BACKEND if BACKEND==\'cuda\' else BACKEND+\'_\'+CPU_GROUP;archive_name=\'reserved_\'+tag+\'.zip\'\n    def export():\n        with zipfile.ZipFile(out/archive_name,\'w\',compression=zipfile.ZIP_DEFLATED) as z:\n            for folder,base,prefix in [(here/\'results\',here,\'\'),(here/\'inner\',here,\'\'),\n                (parent/\'submissions/2025-03-stage2\',parent/\'submissions/2025-03-stage2\',\'march/\')]:\n                for p in folder.rglob(\'*\'):\n                    if p.is_file():z.write(p,prefix+p.relative_to(base).as_posix())\n    try:\n        for model in models:\n            for fold in frozen[\'reserved_months\']:\n                for config in dict.fromkeys([\'baseline\',\'previous\',frozen[\'individual\'][model]]):execute(model,fold,config,threads=4)\n                full_configs=[\'baseline\',\'previous\'] if fold==\'2024-11\' or model==MODELS[-1] else [\'baseline\']\n                for config in full_configs:execute(model,fold,config,\'full\',threads=4)\n                export()\n        # Recreate the exact original March split on this host, before any fit.\n        full_cache=cache/\'final_2025-03\';full_cache.mkdir(exist_ok=True)\n        raw=read_raw(recipe)\n        with warnings.catch_warnings():\n            warnings.simplefilter(\'ignore\',pd.errors.PerformanceWarning)\n            original=recipe.build_retail_features_with_split(raw,split_mode=\'train_val_test\',min_year=2023,\n                val_year=2025,val_month=2,test_year=2025,test_month=3,create_test_if_missing=True,\n                expected_test_size=18657,use_val_as_test_history=True,base_feature_kwargs=recipe.COMMON_BASE_FEATURE_KWARGS,verbose=False)\n        compact=compact_fold(recipe,original);del original;gc.collect()\n        for key in [\'test_train_features\',\'test_features\']:compact[key].to_parquet(full_cache/(key+\'.parquet\'),index=False)\n        assert compact[\'test_train_features\'].year_month_index.max()==25 and compact[\'test_features\'].pto.isna().all()\n        (full_cache/\'manifest.json\').write_text(json.dumps(dict(categorical_cols=compact[\'categorical_cols\'],feature_cols=compact[\'feature_cols\'],\n            notebook_sha256=NOTEBOOK_SHA256,raw_sha256=hashlib.sha256((raw_dir/\'train_2.csv\').read_bytes()).hexdigest()),indent=2))\n        fingerprints={key:hashlib.sha256(pd.util.hash_pandas_object(compact[key],index=False).values.tobytes()).hexdigest() for key in [\'test_train_features\',\'test_features\']}\n        extra=raw[(raw.year==2025)&(raw.month==2)].copy();extra[\'month\']=3;extra[\'pto\']=np.nan\n        legacy,_=namespace[\'spatial_table\'](pd.concat([raw,extra],ignore_index=True),kind=\'exp50\')\n        legacy[expected[\'columns\']].to_parquet(full_cache/\'legacy_features.parquet\',index=False)\n        del raw,extra,legacy,compact;gc.collect()\n        from march_forecast import run\n        composition=frozen[\'ensemble\'][\'composition\']\n        for model in models:\n            run(model,threads=4)\n            extra_configs=[]\n            if composition in [\'baseline\',\'cautious_new\']:extra_configs=[\'baseline\']\n            elif composition==\'previous\':extra_configs=[\'previous\']\n            elif composition==\'mandatory_mixed\':extra_configs=[\'previous\' if model==MODELS[-1] else \'baseline\']\n            for config in extra_configs:run(model,threads=4,configuration=config)\n            export()\n        check=dict(status=\'passed\',backend=BACKEND,models=models,original_legacy_exact=True,old_future_checks=checks,\n            frozen_selection_sha256=hashlib.sha256((here/\'frozen_final_selection.json\').read_bytes()).hexdigest(),\n            canonical_features_sha256=hashlib.sha256((new_cache/\'features.parquet\').read_bytes()).hexdigest(),\n            original_march_frame_fingerprints=fingerprints,\n            environment={n:importlib.metadata.version(n) for n in [\'numpy\',\'pandas\',\'scipy\',\'scikit-learn\',\'catboost\',\'lightgbm\',\'xgboost\']})\n        (out/(\'reserved_\'+tag+\'_checks.json\')).write_text(json.dumps(check,indent=2))\n    finally:export()\n    print(\'FROZEN_RESERVED_AND_MARCH_DONE\',BACKEND,flush=True)\n\nif __name__==\'__main__\':main()\n',encoding='utf-8')
subprocess.check_call([sys.executable,'-u',str(p)])
